In [ ]:
from pathlib import Path
import os
import sys

import torch
import numpy as np
import json

from groundingdino.util.inference import load_model, load_image, predict, annotate

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.schemas import Box3D
from src.common.nuscenes_utils import (
    convert_global_bbox_to_ego,
    make_box_corners_ego,
    transform_ego_to_camera,
    project_camera_points,
    filter_points_in_image
)

# Category name conversion dictionary
CATEGORY_CONVERSION = {
    'noise': 'noise',
    'animal': 'animal',
    'human.pedestrian.adult': 'person',
    'human.pedestrian.child': 'person',
    'human.pedestrian.construction_worker': 'person',
    'human.pedestrian.personal_mobility': 'person',
    'human.pedestrian.police_officer': 'person',
    'human.pedestrian.stroller': 'person',
    'human.pedestrian.wheelchair': 'person',
    'movable_object.barrier': 'barrier',
    'movable_object.debris': 'debris',
    'movable_object.pushable_pullable': 'pushable_pullable',
    'movable_object.trafficcone': 'trafficcone',
    'static_object.bicycle_rack': 'bicycle_rack',
    'vehicle.bicycle': 'bicycle',
    'vehicle.bus.bendy': 'bus',
    'vehicle.bus.rigid': 'bus',
    'vehicle.car': 'car',
    'vehicle.construction': 'construction_vehicle',
    'vehicle.emergency.ambulance': 'ambulance',
    'vehicle.emergency.police': 'police_car',
    'vehicle.motorcycle': 'motorcycle',
    'vehicle.trailer': 'trailer',
    'vehicle.truck': 'truck',
    'flat.driveable_surface': 'driveable_surface',
    'flat.other': 'other',
    'flat.sidewalk': 'sidewalk',
    'flat.terrain': 'terrain',
    'static.manmade': 'manmade',
    'static.other': 'other',
    'static.vegetation': 'vegetation',
    'vehicle.ego': 'car'
}
# Resolve paths relative to this notebook directory
CONFIG_PATH = ROOT / "GroundingDINO" / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
WEIGHTS_PATH = ROOT / "GroundingDINO" / "weights/groundingdino_swinb_cogcoor.pth"
IMAGE_PATH = ROOT / "GroundingDINO" / ".asset/cat_dog.jpeg"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Create the model and load the weights
model = load_model(str(CONFIG_PATH), str(WEIGHTS_PATH), device=device)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-mini"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "category.json") as f:
    categories = json.load(f)
# Create hash maps for token lookup
category_lookup = {category["name"]: category["token"] for category in categories}
category_dict = {category["token"]: category["name"] for category in categories}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors}
print(f"category_names: {[category['name'] for category in categories]}")
print(f"sensor_channels: {[sensor['channel'] for sensor in sensors]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")
# Validate the category conversion dictionary
for category in CATEGORY_CONVERSION.keys():
    if category not in category_lookup:
        raise ValueError(f"Category '{category}' not found in nuScenes categories.")

In [ ]:
# Select the scenes and camera channel
SCENE_NAME = "scene-0061"
CAMERA_CHANNEL = "CAM_FRONT"

scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
# Select the samples
samples = [sample for sample in samples_all if sample["scene_token"] == scene["token"]]
print(f"num_samples: {len(samples)}")
sample_tokens = set(sample["token"] for sample in samples)
# Select the sample_data, ego_poses, and calibrated_sensors in the scene
sample_data = {sd["token"]: sd for sd in sample_data_all if sd["sample_token"] in sample_tokens}
sample_data = {sd_token: sd for sd_token, sd in sample_data.items() if sd["is_key_frame"]}  # Filter by is_key_frame
sample_data = dict(sorted(sample_data.items(), key=lambda item: item[1]["timestamp"]))  # sort by timestamp
ego_pose_tokens = set(sd["ego_pose_token"] for sd in sample_data.values())
ego_poses = {ep["token"]: ep for ep in ego_poses_all if ep["token"] in ego_pose_tokens}
calibrated_sensor_tokens = set(sd["calibrated_sensor_token"] for sd in sample_data.values())
calibrated_sensors = {cs["token"]: cs for cs in calibrated_sensors_all if cs["token"] in calibrated_sensor_tokens}
# Select the sample_annotations and instances in the scene
sample_annotations = {sa["token"]: sa for sa in sample_annotations_all if sa["sample_token"] in sample_tokens}
instance_tokens = set(sa["instance_token"] for sa in sample_annotations.values())
instances = {inst["token"]: inst for inst in instances_all if inst["token"] in instance_tokens}
# Create tracking_ids from instance tokens
tracking_ids = {inst_token: i for i, inst_token in enumerate(instance_tokens)}

# Filter sample_data to only include the selected camera channel
calibrated_sensors = {cs_token: cs for cs_token, cs in calibrated_sensors.items() if cs["sensor_token"] == sensor_lookup[CAMERA_CHANNEL]}
sample_data = {sd_token: sd for sd_token, sd in sample_data.items() if sd["calibrated_sensor_token"] in calibrated_sensors}
ego_poses = {ep_token: ep for ep_token, ep in ego_poses.items() if ep_token in set(sd["ego_pose_token"] for sd in sample_data.values())}

# Show the first five sample data entries for the selected scene
for i, (token, sd) in enumerate(sample_data.items()):
    if i >= 5:
        break
    image_path = NUSCENES_ROOT / sd["filename"]
    ego_pose = ego_poses[sd["ego_pose_token"]]
    camera_translation = calibrated_sensors[sd["calibrated_sensor_token"]]["translation"]
    camera_rotation = calibrated_sensors[sd["calibrated_sensor_token"]]["rotation"]
    camera_intrinsic = calibrated_sensors[sd["calibrated_sensor_token"]]["camera_intrinsic"]
    # Getting bounding boxes in the sample
    annotations_in_sample = [sa for sa in sample_annotations.values() if sa["sample_token"] == sd["sample_token"]]
    boxes_3d = [Box3D.from_dimensions(
        center=np.array(sa["translation"]),
        length=sa["size"][1],
        width=sa["size"][0],
        height=sa["size"][2],
        rotation=np.array(sa["rotation"]),
        label=CATEGORY_CONVERSION[category_dict[instances[sa["instance_token"]]["category_token"]]],
        track_id=tracking_ids[sa["instance_token"]]
    ) for sa in annotations_in_sample]
    print(f"Number of boxes in sample {i}: {len(boxes_3d)}")
    # Convert the boxes to ego vehicle coordinates
    boxes_3d_ego = [convert_global_bbox_to_ego(box, ego_pose["translation"], ego_pose["rotation"]) for box in boxes_3d]
    # Keep only box centers in front of the camera (z > near_plane).
    center_points_ego = np.array([box.center for box in boxes_3d_ego]).T
    center_points_cam = transform_ego_to_camera(center_points_ego, 
                                                camera_translation,
                                                camera_rotation)
    front_center_points_image, valid = project_camera_points(center_points_cam, camera_intrinsic)
    front_boxes_3d_ego = [box for box, v in zip(boxes_3d_ego, valid) if v]
    print(f"Number of front boxes in sample {i}: {len(front_boxes_3d_ego)}")
    # Keep only boxes that are in the camera's field of view
    box_corners_ego = [make_box_corners_ego(box) for box in front_boxes_3d_ego]